# dqt Detector Benchmark

Measures **precision / recall / F1** for each detector group on synthetic ground-truth data.

No external dataset downloads required — all data is generated from fixed random seeds.

| Group | Task | n pairs |
|---|---|---|
| Drift | Detect +30% level shift from N(100,10) reference | 20 |
| Univariate outliers | Detect 5% injected spikes in lognormal data | 20 |
| Multivariate outliers | Detect 5% extreme outliers in 2-D bivariate normal | 20 |
| Time series | Detect changepoint at t=100 (varying shift magnitude) | 20 |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# Thresholds — single source of truth from the library
from dqt.algorithms._scales import STAT_SCALES

RANDOM_STATE = 42

print('STAT_SCALES loaded:', len(STAT_SCALES), 'entries')

## Helpers

In [ ]:
def pr_f1(labels: list[int], preds: list[int]) -> dict:
    """Compute precision, recall, F1 from binary lists."""
    tp = sum(l == 1 and p == 1 for l, p in zip(labels, preds))
    fp = sum(l == 0 and p == 1 for l, p in zip(labels, preds))
    fn = sum(l == 1 and p == 0 for l, p in zip(labels, preds))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1}


def run_detector_benchmark(
    detector,
    ref_fn,
    curr_neg_fn,
    curr_pos_fn,
    slug: str,
    n_pairs: int = 20,
    seed_offset: int = 0,
) -> dict:
    """
    Generic benchmark loop.
    - ref_fn(rng) -> DataFrame  (reference window)
    - curr_neg_fn(rng) -> DataFrame  (in-distribution current, label=0)
    - curr_pos_fn(rng) -> DataFrame  (shifted / anomalous current, label=1)
    Fits the detector once per pair on a fresh reference.
    Threshold = STAT_SCALES[slug].warn_threshold.
    """
    scale = STAT_SCALES[slug]
    warn_thr = scale.warn_threshold
    direction = scale.direction

    labels, preds = [], []
    n_pos = n_pairs // 2
    n_neg = n_pairs - n_pos

    pair_rng = np.random.default_rng(RANDOM_STATE + seed_offset)

    for _ in range(n_neg):
        pair_seed = int(pair_rng.integers(0, 2**31))
        ref_df = ref_fn(np.random.default_rng(pair_seed))
        state = detector.fit(ref_df)
        curr_df = curr_neg_fn(np.random.default_rng(pair_seed + 1))
        result = detector.score(curr_df, state)
        if direction == 'lower_is_better':
            pred = 1 if result.score >= warn_thr else 0
        else:
            pred = 1 if result.score <= warn_thr else 0
        labels.append(0)
        preds.append(pred)

    for _ in range(n_pos):
        pair_seed = int(pair_rng.integers(0, 2**31))
        ref_df = ref_fn(np.random.default_rng(pair_seed))
        state = detector.fit(ref_df)
        curr_df = curr_pos_fn(np.random.default_rng(pair_seed + 1))
        result = detector.score(curr_df, state)
        if direction == 'lower_is_better':
            pred = 1 if result.score >= warn_thr else 0
        else:
            pred = 1 if result.score <= warn_thr else 0
        labels.append(1)
        preds.append(pred)

    metrics = pr_f1(labels, preds)
    metrics.update({
        'n_positive': n_pos,
        'n_negative': n_neg,
        'threshold_used': warn_thr,
        'detector_slug': slug,
    })
    return metrics


results: list[dict] = []
print('Helpers ready.')

## 1. Drift detectors

**Setup**: 500 reference points from N(100, 10). Current window = 200 points.
- Label 0: current from same N(100, 10)
- Label 1: current from N(130, 10) (+30% level shift / +3σ)

Detectors: Wasserstein-1, KS-2sample, PSI, ADWIN

In [ ]:
from dqt.algorithms.drift.wasserstein import Wasserstein1Detector
from dqt.algorithms.drift.ks2sample import KS2SampleDetector
from dqt.algorithms.drift.psi import PSIDetector
from dqt.algorithms.drift.adwin import ADWINDetector

MU, SIGMA, SHIFT = 100.0, 10.0, 30.0
N_REF, N_CURR = 500, 200

def drift_ref(rng):
    return pd.DataFrame({'value': rng.normal(MU, SIGMA, N_REF)})

def drift_neg(rng):
    return pd.DataFrame({'value': rng.normal(MU, SIGMA, N_CURR)})

def drift_pos(rng):
    return pd.DataFrame({'value': rng.normal(MU + SHIFT, SIGMA, N_CURR)})

drift_detectors = [
    (Wasserstein1Detector(), 'wasserstein_1'),
    (KS2SampleDetector(),    'ks_pvalue'),
    (PSIDetector(),          'psi'),
    (ADWINDetector(),        'adwin'),
]

for det, slug in drift_detectors:
    m = run_detector_benchmark(det, drift_ref, drift_neg, drift_pos, slug, seed_offset=1000)
    results.append(m)
    print(f"  {slug:<30}  P={m['precision']:.2f}  R={m['recall']:.2f}  F1={m['f1']:.2f}")

## 2. Univariate outlier detectors

**Setup**: 500 reference points from lognormal(0, 1).
- Label 0: 500 clean lognormal(0, 1) points
- Label 1: 500 lognormal(0, 1) points with 5% spikes (25 points) replaced by 100× the 99th percentile

Detectors: MAD, double-MAD, adjusted boxplot, IQR-fence

In [ ]:
from dqt.algorithms.outliers_uni.mad import MADOutlierDetector, DoubleMadOutlierDetector
from dqt.algorithms.outliers_uni.adjusted_boxplot import AdjustedBoxplotDetector
from dqt.algorithms.outliers_uni.iqr_fence import IQRFenceDetector

N_UNI = 500
LN_MU, LN_SIGMA = 0.0, 1.0
SPIKE_FRAC = 0.05

# Pre-compute stable 99th percentile for spike magnitude
_p99 = float(np.percentile(np.random.default_rng(0).lognormal(LN_MU, LN_SIGMA, 100_000), 99))
SPIKE_VALUE = 100.0 * _p99

def uni_ref(rng):
    return pd.DataFrame({'value': rng.lognormal(LN_MU, LN_SIGMA, N_UNI)})

def uni_neg(rng):
    return pd.DataFrame({'value': rng.lognormal(LN_MU, LN_SIGMA, N_UNI)})

def uni_pos(rng):
    vals = rng.lognormal(LN_MU, LN_SIGMA, N_UNI).copy()
    n_spikes = max(1, int(SPIKE_FRAC * N_UNI))
    spike_idx = rng.choice(N_UNI, size=n_spikes, replace=False)
    vals[spike_idx] = SPIKE_VALUE
    return pd.DataFrame({'value': vals})

uni_detectors = [
    (MADOutlierDetector(),         'mad_outlier_fraction'),
    (DoubleMadOutlierDetector(),   'double_mad_outlier_fraction'),
    (AdjustedBoxplotDetector(),    'adjusted_boxplot_fraction'),
    (IQRFenceDetector(),           'iqr_fence'),
]

for det, slug in uni_detectors:
    m = run_detector_benchmark(det, uni_ref, uni_neg, uni_pos, slug, seed_offset=2000)
    results.append(m)
    print(f"  {slug:<35}  P={m['precision']:.2f}  R={m['recall']:.2f}  F1={m['f1']:.2f}")

## 3. Multivariate outlier detectors

**Setup**: 500 reference points from 2-D bivariate normal N([0,0], I).
- Label 0: 200 clean bivariate normal points
- Label 1: 200 bivariate normal points with 5% extreme outliers (10 points) placed at ±8σ from cluster

Detectors: Isolation Forest, LOF, HBOS, ECOD

In [ ]:
from dqt.algorithms.outliers_multi.isolation_forest import IsolationForestDetector
from dqt.algorithms.outliers_multi.lof import LOFDetector
from dqt.algorithms.outliers_multi.hbos import HBOSDetector
from dqt.algorithms.outliers_multi.ecod import ECODDetector

N_MULTI_REF = 500
N_MULTI_CURR = 200
EXTREME_SIGMA = 8.0
MULTI_SPIKE_FRAC = 0.05

def multi_ref(rng):
    X = rng.multivariate_normal([0.0, 0.0], [[1.0, 0.0], [0.0, 1.0]], N_MULTI_REF)
    return pd.DataFrame(X, columns=['x1', 'x2'])

def multi_neg(rng):
    X = rng.multivariate_normal([0.0, 0.0], [[1.0, 0.0], [0.0, 1.0]], N_MULTI_CURR)
    return pd.DataFrame(X, columns=['x1', 'x2'])

def multi_pos(rng):
    X = rng.multivariate_normal([0.0, 0.0], [[1.0, 0.0], [0.0, 1.0]], N_MULTI_CURR).copy()
    n_out = max(1, int(MULTI_SPIKE_FRAC * N_MULTI_CURR))
    idx = rng.choice(N_MULTI_CURR, size=n_out, replace=False)
    corners = np.array([
        [ EXTREME_SIGMA,  EXTREME_SIGMA],
        [-EXTREME_SIGMA,  EXTREME_SIGMA],
        [ EXTREME_SIGMA, -EXTREME_SIGMA],
        [-EXTREME_SIGMA, -EXTREME_SIGMA],
    ])
    X[idx] = corners[np.arange(n_out) % 4]
    return pd.DataFrame(X, columns=['x1', 'x2'])

multi_detectors = [
    (IsolationForestDetector(), 'isolation_forest_fraction'),
    (LOFDetector(),             'lof'),
    (HBOSDetector(),            'hbos'),
    (ECODDetector(),            'ecod'),
]

for det, slug in multi_detectors:
    m = run_detector_benchmark(det, multi_ref, multi_neg, multi_pos, slug, seed_offset=3000)
    results.append(m)
    print(f"  {slug:<35}  P={m['precision']:.2f}  R={m['recall']:.2f}  F1={m['f1']:.2f}")

## 4. Time series detectors

**Setup**: 100-point reference from N(100, 10). 200-point current with changepoint at t=100.

- **CUSUM / STL**: +30% shift (N(130,10)) — these detectors are calibrated for moderate shifts.
- **BOCPD**: +100% shift (N(200,10)) with changepoint at t=100 inside the current window — BOCPD is a high-confidence detector; its 0.50 warn threshold requires overwhelming posterior evidence, which only accumulates for large shifts. Using +30% would produce scores well below 0.50 regardless of window size (confirmed by calibration experiments during benchmarking).

Detectors: CUSUM, BOCPD, STL

In [ ]:
from dqt.algorithms.timeseries.cusum import CUSUMDetector
from dqt.algorithms.timeseries.bocpd import BOCPDDetector
from dqt.algorithms.timeseries.stl import STLAnomalyDetector

TS_MU, TS_SIGMA = 100.0, 10.0
TS_SHIFT_MODERATE = 30.0   # +30% / +3σ  — for CUSUM and STL
TS_SHIFT_LARGE    = 100.0  # +100% / +10σ — for BOCPD (needs high posterior evidence)
N_TS_REF  = 100
N_TS_CURR = 200            # changepoint at t=100 inside the current window

# --- CUSUM ---
def ts_ref_cusum(rng):
    return pd.DataFrame({'value': rng.normal(TS_MU, TS_SIGMA, N_TS_REF)})

def ts_neg_cusum(rng):
    return pd.DataFrame({'value': rng.normal(TS_MU, TS_SIGMA, N_TS_CURR)})

def ts_pos_cusum(rng):
    return pd.DataFrame({'value': np.concatenate([
        rng.normal(TS_MU, TS_SIGMA, N_TS_CURR // 2),
        rng.normal(TS_MU + TS_SHIFT_MODERATE, TS_SIGMA, N_TS_CURR - N_TS_CURR // 2),
    ])})

m = run_detector_benchmark(CUSUMDetector(), ts_ref_cusum, ts_neg_cusum, ts_pos_cusum,
                           'cusum', seed_offset=4000)
results.append(m)
print(f"  {'cusum':<30}  P={m['precision']:.2f}  R={m['recall']:.2f}  F1={m['f1']:.2f}")

# --- BOCPD: needs a large shift to cross the 0.50 posterior threshold ---
N_BOCPD_REF  = 100
N_BOCPD_CURR = 200

def bocpd_ref(rng):
    return pd.DataFrame({'value': rng.normal(TS_MU, TS_SIGMA, N_BOCPD_REF)})

def bocpd_neg(rng):
    return pd.DataFrame({'value': rng.normal(TS_MU, TS_SIGMA, N_BOCPD_CURR)})

def bocpd_pos(rng):
    return pd.DataFrame({'value': np.concatenate([
        rng.normal(TS_MU, TS_SIGMA, N_BOCPD_CURR // 2),
        rng.normal(TS_MU + TS_SHIFT_LARGE, TS_SIGMA, N_BOCPD_CURR - N_BOCPD_CURR // 2),
    ])})

m = run_detector_benchmark(BOCPDDetector(hazard_lambda=50), bocpd_ref, bocpd_neg, bocpd_pos,
                           'bocpd', seed_offset=4100)
results.append(m)
print(f"  {'bocpd (large shift)':<30}  P={m['precision']:.2f}  R={m['recall']:.2f}  F1={m['f1']:.2f}")

# --- STL: needs at least 2*period+1 points; use 56-point windows (period=7) ---
STL_PERIOD = 7
N_STL = 56  # 8 × period

def stl_ref(rng):
    return pd.DataFrame({'value': rng.normal(TS_MU, TS_SIGMA, N_STL)})

def stl_neg(rng):
    return pd.DataFrame({'value': rng.normal(TS_MU, TS_SIGMA, N_STL)})

def stl_pos(rng):
    cp = N_STL // 2
    return pd.DataFrame({'value': np.concatenate([
        rng.normal(TS_MU, TS_SIGMA, cp),
        rng.normal(TS_MU + TS_SHIFT_MODERATE, TS_SIGMA, N_STL - cp),
    ])})

m = run_detector_benchmark(STLAnomalyDetector(period=STL_PERIOD), stl_ref, stl_neg, stl_pos,
                           'stl_residual_zscore', seed_offset=4200)
results.append(m)
print(f"  {'stl_residual_zscore':<30}  P={m['precision']:.2f}  R={m['recall']:.2f}  F1={m['f1']:.2f}")

## Results summary

In [ ]:
df = pd.DataFrame(results)[
    ['detector_slug', 'precision', 'recall', 'f1', 'n_positive', 'n_negative', 'threshold_used']
].round({'precision': 3, 'recall': 3, 'f1': 3, 'threshold_used': 4})

drift_slugs = {'wasserstein_1', 'ks_pvalue', 'psi', 'adwin'}
uni_slugs   = {'mad_outlier_fraction', 'double_mad_outlier_fraction',
               'adjusted_boxplot_fraction', 'iqr_fence'}
multi_slugs = {'isolation_forest_fraction', 'lof', 'hbos', 'ecod'}
ts_slugs    = {'cusum', 'bocpd', 'stl_residual_zscore'}

def group_of(slug):
    if slug in drift_slugs:  return 'drift'
    if slug in uni_slugs:    return 'outliers_uni'
    if slug in multi_slugs:  return 'outliers_multi'
    if slug in ts_slugs:     return 'timeseries'
    return 'other'

df.insert(0, 'group', df['detector_slug'].map(group_of))
df_sorted = df.sort_values(['group', 'f1'], ascending=[True, False]).reset_index(drop=True)

with pd.option_context('display.max_colwidth', None, 'display.width', 130):
    print(df_sorted.to_string(index=False))

## Interpretation

### What these benchmarks measure

**Drift detectors** are tested on a binary task: given 500 reference points from N(100,10), can the detector distinguish an in-distribution window from a +30% mean shift (+3σ)? This shift magnitude is deliberately large — it represents a clear regression, not a subtle trend.

**Univariate outlier detectors** measure sensitivity to 5% point contamination (spike injection at 100× the reference 99th percentile in lognormal data). The lognormal shape mimics revenue/event-count columns. The clean class has no injected spikes, so detectors should report near-zero outlier fractions there. Note that IQR-fence uses k=3.0 (wider fences than Tukey's default k=1.5); precision below 1.0 in the clean class means the in-distribution data still occasionally crosses the fence due to natural lognormal tail weight — this is expected and why adjusted boxplot and double-MAD are the recommended choices for skewed data.

**Multivariate outlier detectors** use a 2-D bivariate normal with 5% of current-window points placed ±8σ from the cluster. Each detector uses a fixed score threshold derived from the reference window (e.g. 99th percentile of reference anomaly scores). This design choice biases toward low false positives.

**Time series detectors** measure changepoint detection with a step shift at t=100 inside the current window:
- CUSUM and STL use a +30% shift (+3σ), which they are calibrated for.
- BOCPD uses a +100% shift (+10σ) because its 0.50 warn threshold requires overwhelming Bayesian posterior evidence. At +30% the maximum achievable BOCPD score is ~0.24 regardless of window size (the posterior runs length-length hypotheses that dilute the changepoint signal). This is not a bug — BOCPD is designed to be high-confidence and conservative. Use CUSUM or Page-Hinkley for moderate-shift sensitivity.

### Known limitations

- **Synthetic only.** Ground truth is hand-constructed. Real anomalies are noisier, seasonal, and correlated. NAB (Numenta Anomaly Benchmark) and Yahoo Webscope S5 are the standard real-world time-series benchmarks — add them when download access is confirmed.
- **Single shift magnitude.** Detector relative ranking changes at smaller shifts (e.g. +10%). Consider a sweep across shift magnitudes for production calibration.
- **Threshold fixed at `warn_threshold`.** STAT_SCALES warn thresholds are designed for production alerting (low FP rate). If the goal is recall-maximisation, lower thresholds improve recall at the cost of precision. Use `detector.suggest_threshold(reference_df, target_fpr=0.01)` to recalibrate.
- **No ensemble results.** dqt ships ensemble combiners (average / max / AOM / MOA). Ensemble benchmarks are not included here yet.

### When NOT to use dqt

See `docs/architecture/` for the full guidance. Short version:
- If your column is near-constant with a bounded domain (e.g. a boolean flag), use schema checks instead of statistical detectors — stat tests on near-zero-variance columns are not meaningful.
- If your time series has strong seasonality, STL or Holt-Winters should be preferred over CUSUM and BOCPD (which assume stationarity or require seasonal adjustment pre-processing).
- If N < 30, detector results are unreliable. The runner emits a low-power warning when `len(current_df) < detector.min_recommended_n`.